In [1]:
datasets_ara=[
    "/kaggle/input/datasets/jorx04/processed-video-dataset-ara",
    "/kaggle/input/datasets/jorx04/processed-video-dataset-ara-2",
    "/kaggle/input/datasets/jousefna/processed-video-dataset-ara-3",
    "/kaggle/input/datasets/jorx04/processed-video-dataset-ara-4"
]

datasets_eng=[
    "/kaggle/input/datasets/jorx04/processed-video-dataset-eng",
    "/kaggle/input/datasets/jorx04/processed-video-dataset-eng-2",
    "/kaggle/input/datasets/jorx04/processed-video-dataset-eng-3"
]

notebooks_ara=[
    "/kaggle/input/notebooks/studentlymailer/specter-ara-1/flame_parameters/",
    "/kaggle/input/notebooks/jorx04/specter-ara-2/flame_parameters/",
    "/kaggle/input/notebooks/studentlymailer/specter-ara-3/flame_parameters/",
    "/kaggle/input/notebooks/jorx04/specter-ara-4/flame_parameters/"
]

notebooks_eng=[
    "/kaggle/input/notebooks/jousefna/specter-eng-1/flame_parameters/",
    "/kaggle/input/notebooks/jorx04/specter-eng-2/flame_parameters/",
    "/kaggle/input/notebooks/jousefna/specter-eng-3/flame_parameters/"
]

In [12]:
!mkdir /tmp/dataset
!mkdir /tmp/dataset/video
!mkdir /tmp/dataset/flame
!mkdir /tmp/dataset/video/ara
!mkdir /tmp/dataset/video/eng
!mkdir /tmp/dataset/flame/ara
!mkdir /tmp/dataset/flame/eng

In [13]:
!dir /tmp/dataset

flame  video


In [14]:
import os
import subprocess

tasks = [
    (datasets_ara, "/tmp/dataset/video/ara"),
    (datasets_eng, "/tmp/dataset/video/eng"),
    (notebooks_ara, "/tmp/dataset/flame/ara"),
    (notebooks_eng, "/tmp/dataset/flame/eng")
]

for sources, dest in tasks:
    os.makedirs(dest, exist_ok=True)
    for src in sources:
        clean_src = src.rstrip('/')
        subprocess.run(["cp", "-a", f"{clean_src}/.", dest], check=True)

In [16]:
import os

def get_expected_video_bases(param_dir):
    if not os.path.exists(param_dir):
        return set()
    
    expected_bases = set()
    for f in os.listdir(param_dir):
        if os.path.isfile(os.path.join(param_dir, f)):
            base_name = os.path.splitext(f)[0]
            if base_name.endswith("_flame_params"):
                clean_name = base_name.replace("_flame_params", "")
                expected_bases.add(clean_name)
            else:
                expected_bases.add(base_name)
    return expected_bases

def remove_unmatched_files(video_dir, param_dir):
    if not os.path.exists(video_dir) or not os.path.exists(param_dir):
        print(f"Skipping: Directory not found ({video_dir} or {param_dir})")
        return 0

    expected_video_bases = get_expected_video_bases(param_dir)
    deleted_count = 0

    for video_file in os.listdir(video_dir):
        video_path = os.path.join(video_dir, video_file)
        if not os.path.isfile(video_path):
            continue

        video_base = os.path.splitext(video_file)[0]
        
        if video_base not in expected_video_bases:
            os.remove(video_path)
            deleted_count += 1

    return deleted_count

videos_ara = "/tmp/dataset/video/ara"
flame_ara = "/tmp/dataset/flame/ara"

videos_eng = "/tmp/dataset/video/eng"
flame_eng = "/tmp/dataset/flame/eng"

print("--- Cleaning Arabic Videos ---")
deleted_ara = remove_unmatched_files(videos_ara, flame_ara)
print(f"Removed {deleted_ara} unmatched files from Videos Ara.")

print("\n--- Cleaning English Videos ---")
deleted_eng = remove_unmatched_files(videos_eng, flame_eng)
print(f"Removed {deleted_eng} unmatched files from Videos Eng.")

--- Cleaning Arabic Videos ---
Removed 236 unmatched files from Videos Ara.

--- Cleaning English Videos ---
Removed 5 unmatched files from Videos Eng.


In [21]:
def count_files(path):
    if not os.path.exists(path):
        return 0
    total = 0
    for root, _, files in os.walk(path):
        total += len(files)
    return total

tasks = [
    ("Ara", "/tmp/dataset/flame/ara", "/tmp/dataset/video/ara"),
    ("Eng", "/tmp/dataset/flame/eng", "/tmp/dataset/video/eng")
]

for name, flame, video in tasks:
    flame_count = count_files(flame)
    video_count = count_files(video)
    
    status = "Match" if flame_count == video_count  else "Mismatch"
    print(f"--- {name} ---")
    print(f"Total flame files: {flame_count}")
    print(f"Total video files: {video_count}")
    print(f"Status: {status}\n")

--- Ara ---
Total flame files: 8527
Total video files: 8527
Status: Match

--- Eng ---
Total flame files: 11055
Total video files: 11055
Status: Match



In [25]:
import os
import subprocess
from concurrent.futures import ProcessPoolExecutor, as_completed
from tqdm import tqdm

def extract_audio(paths):
    input_path, output_path = paths
    subprocess.run([
        "ffmpeg", "-i", input_path,
        "-vn", "-acodec", "pcm_s16le", 
        "-ar", "16000", "-ac", "1", 
        output_path, "-y"
    ], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

extraction_tasks = [
    ("/tmp/dataset/video/ara", "/tmp/dataset/audio/ara"),
    ("/tmp/dataset/video/eng", "/tmp/dataset/audio/eng")
]

all_files = []
for video_dir, audio_dir in extraction_tasks:
    if not os.path.exists(video_dir): continue
    os.makedirs(audio_dir, exist_ok=True)
    for f in os.listdir(video_dir):
        if f.endswith(".mp4"):
            all_files.append((
                os.path.join(video_dir, f), 
                os.path.join(audio_dir, os.path.splitext(f)[0] + ".wav")
            ))

with ProcessPoolExecutor() as executor:
    futures = [executor.submit(extract_audio, task) for task in all_files]
    
    for _ in tqdm(as_completed(futures), total=len(all_files), desc="Overall Progress"):
        pass

Overall Progress: 100%|██████████| 19582/19582 [19:43<00:00, 16.55it/s]


In [37]:
tasks = [
    ("Ara", "/tmp/dataset/flame/ara", "/tmp/dataset/audio/ara", "/tmp/dataset/video/ara"),
    ("Eng", "/tmp/dataset/flame/eng", "/tmp/dataset/audio/eng", "/tmp/dataset/video/eng")
]

for name, flame, audio, video in tasks:
    flame_count = count_files(flame)
    audio_count = count_files(audio)
    video_count = count_files(video)
    
    status = "Match" if flame_count == audio_count == video_count  else "Mismatch"
    print(f"--- {name} ---")
    print(f"Total flame files: {flame_count}")
    print(f"Total audio files: {audio_count}")
    print(f"Total video files: {video_count}")
    print(f"Status: {status}\n")

--- Ara ---
Total flame files: 8527
Total audio files: 8527
Total video files: 8527
Status: Match

--- Eng ---
Total flame files: 11055
Total audio files: 11055
Total video files: 11055
Status: Match



In [44]:
import os
import json
import subprocess
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
username = user_secrets.get_secret("KAGGLE_USERNAME")
os.environ["KAGGLE_USERNAME"] = username
os.environ["KAGGLE_KEY"] = user_secrets.get_secret("KAGGLE_KEY")

dataset_dir = "/tmp/dataset"
dataset_title = "fully collected dataset"
dataset_id = f"{username}/fully-collected-dataset"

metadata = {
    "title": dataset_title,
    "id": dataset_id,
    "licenses": [{"name": "CC0-1.0"}],
    "isPrivate": False
}

metadata_path = os.path.join(dataset_dir, "dataset-metadata.json")
with open(metadata_path, "w") as f:
    json.dump(metadata, f, indent=4)

print(f"Preparing to upload '{dataset_title}' from {dataset_dir}...")

try:
    subprocess.run(["kaggle", "datasets", "create", "-p", dataset_dir, "--dir-mode", "zip"], check=True)
    print("Dataset successfully created and made public!")
    
except subprocess.CalledProcessError:
    print("Dataset ID already exists. Pushing a new version instead...")
    subprocess.run([
        "kaggle", "datasets", "version", 
        "-p", dataset_dir, 
        "-m", "Initial upload of full compiled dataset",
        "--dir-mode", "zip"
    ], check=True)
    print("Dataset version successfully updated!")

Preparing to upload 'fully collected dataset' from /tmp/dataset...
Starting upload for file video.zip


100%|██████████| 16.5G/16.5G [04:03<00:00, 72.8MB/s]


Upload successful: video.zip (16GB)
Starting upload for file flame.zip


100%|██████████| 1.28G/1.28G [00:19<00:00, 71.9MB/s]


Upload successful: flame.zip (1GB)
Starting upload for file audio.zip


100%|██████████| 2.42G/2.42G [00:32<00:00, 78.7MB/s]


Upload successful: audio.zip (2GB)
Your private Dataset is being created. Please check progress at https://www.kaggle.com/datasets/jorx04/fully-collected-dataset
Dataset successfully created and made public!
